# Cross-source analysis — incidents x telemetry x maintenance

Combines all sources on the universal key `machine_id` (and `incident_id`
for the maintenance link), reusing `src/analyses/`.

- **Section A** reproduces the standard cross-source graphs via `src/`.
- **Section B** is exploratory (inline) to think about deeper relationships.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

%matplotlib inline

from src import config
print("Artifacts dir:", config.CROSS_SOURCE_ARTIFACTS_DIR)

## 1. Generate a fresh run (or pick the latest)

- `GENERATE_NEW_RUN = True` -> run the cross-source analysis.
- `GENERATE_NEW_RUN = False` -> only explore the most recent existing run.

In [ ]:
from src.analyses.runner import run_default

GENERATE_NEW_RUN = False  # True = create a new run; False = explore the latest

if GENERATE_NEW_RUN:
    run_dir = run_default()
else:
    runs = sorted(
        p for p in config.CROSS_SOURCE_ARTIFACTS_DIR.iterdir()
        if p.is_dir() and p.name.isdigit()
    )
    assert runs, "No cross-source run found. Set GENERATE_NEW_RUN = True to create one."
    run_dir = runs[-1]

RUN_ID = run_dir.name
print("Active run:", RUN_ID)

## 2. Build the joined tables (via `src/analyses/`)

`load_sources` reuses each source's loader; the builders join everything
into the per-machine profile and the reactive↔incident table.

In [ ]:
from src.analyses.joins import (
    load_sources,
    build_machine_profile,
    build_reactive_incident_join,
)

sources = load_sources()
profile = build_machine_profile(sources)
reactive = build_reactive_incident_join(sources)
profile

## A. Standard cross-source graphs (via `src/`)

The exact functions used at each run; displays the saved PNGs.

In [ ]:
from src.analyses import plots

for png_path in plots.plot_all(profile, reactive, run_dir):
    display(Image(filename=str(png_path)))

## B. Exploratory analyses (inline)

Open questions to think about — promote to `src/analyses/` if they prove useful.

### B.1 Correlation matrix of the machine profile

In [ ]:
num = profile.select_dtypes('number')
corr = num.corr()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=6)
ax.set_title("Machine profile — correlation matrix")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

### B.2 Delay between an incident and its reactive maintenance

How fast are reactive maintenances carried out after the triggering incident?

In [ ]:
inc_dates = sources['incidents'][[config.ID_COLUMN, config.DATE_COLUMN]]
maint = sources['maintenance']
reac = maint[
    (maint[config.MAINTENANCE_TYPE_COLUMN] == 'reactive')
    & maint[config.MAINTENANCE_INCIDENT_COLUMN].notna()
]
j = reac.merge(
    inc_dates,
    left_on=config.MAINTENANCE_INCIDENT_COLUMN,
    right_on=config.ID_COLUMN,
    how='inner',
)
delay_days = (
    j[config.MAINTENANCE_TIMESTAMP_COLUMN].dt.tz_localize(None) - j[config.DATE_COLUMN]
).dt.days
print(delay_days.describe().round(1))

ax = delay_days.plot.hist(bins=20, color="#4C72B0", edgecolor="white", figsize=(9, 4))
ax.set_xlabel("Delay incident -> reactive maintenance (days)")
ax.set_title("Reaction time for reactive maintenance")
plt.tight_layout()
plt.show()

### B.3 Declared criticality vs observed load

Does the declared criticality match the real incident / maintenance load?

In [ ]:
by_crit = (
    profile.groupby(config.MACHINE_CRITICALITY_COLUMN)[
        ['n_incidents', 'n_maintenance', 'n_reactive', 'mean_severity']
    ]
    .mean()
    .reindex(['LOW', 'MEDIUM', 'HIGH'])
    .round(2)
)
display(by_crit)

## C. Promoting an analysis to production

When a Section B cell proves useful, move its logic into `src/analyses/`
(joins + plots) and wire it into `runner.py` so it becomes a standard artifact.